In [1]:
# This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment2/'
FOLDERNAME = "Code/ML/Stanford_CNN_2023/assignment2/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Code/ML/Stanford_CNN_2023/assignment2/cs231n/datasets
/content/drive/My Drive/Code/ML/Stanford_CNN_2023/assignment2


# Introduction to PyTorch

You've written a lot of code in this assignment to provide a whole host of neural network functionality. Dropout, Batch Norm, and 2D convolutions are some of the workhorses of deep learning in computer vision. You've also worked hard to make your code efficient and vectorized.

For the last part of this assignment, though, we're going to leave behind your beautiful codebase and instead migrate to one of two popular deep learning frameworks: in this instance, PyTorch.

## Why do we use deep learning frameworks?

* Our code will now run on GPUs! This will allow our models to train much faster. When using a framework like PyTorch you can harness the power of the GPU for your own custom neural network architectures without having to write CUDA code directly (which is beyond the scope of this class).
* In this class, we want you to be ready to use one of these frameworks for your project so you can experiment more efficiently than if you were writing every feature you want to use by hand.
* We want you to stand on the shoulders of giants! PyTorch is an excellent frameworks that will make your lives a lot easier, and now that you understand their guts, you are free to use them :)
* Finally, we want you to be exposed to the sort of deep learning code you might run into in academia or industry.

## What is PyTorch?

PyTorch is a system for executing dynamic computational graphs over Tensor objects that behave similarly as numpy ndarray. It comes with a powerful automatic differentiation engine that removes the need for manual back-propagation.

## How do I learn PyTorch?

One of our former instructors, Justin Johnson, made an excellent [tutorial](https://github.com/jcjohnson/pytorch-examples) for PyTorch.

You can also find the detailed [API doc](http://pytorch.org/docs/stable/index.html) here. If you have other questions that are not addressed by the API docs, the [PyTorch forum](https://discuss.pytorch.org/) is a much better place to ask than StackOverflow.

# Table of Contents

This assignment has 5 parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final project.

1. Part I, Preparation: we will use CIFAR-10 dataset.
2. Part II, Barebones PyTorch: **Abstraction level 1**, we will work directly with the lowest-level PyTorch Tensors.
3. Part III, PyTorch Module API: **Abstraction level 2**, we will use `nn.Module` to define arbitrary neural network architecture.
4. Part IV, PyTorch Sequential API: **Abstraction level 3**, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.
5. Part V, CIFAR-10 open-ended challenge: please implement your own network to get as high accuracy as possible on CIFAR-10. You can experiment with any layer, optimizer, hyperparameters or other advanced features.

Here is a table of comparison:

| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |

# GPU

You can manually switch to a GPU device on Colab by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. You should do this before running the following cells to import packages, since the kernel gets restarted upon switching runtimes.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

using device: cuda


# Part I. Preparation

Now, let's load the CIFAR-10 dataset. This might take a couple minutes the first time you do it, but the files should stay cached after that.

In previous parts of the assignment we had to write our own code to download the CIFAR-10 dataset, preprocess it, and iterate through it in minibatches; PyTorch provides convenient tools to automate this process for us.

In [ ]:
NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data
# and for performing data augmentation; here we set up a transform to
# preprocess the data by subtracting the mean RGB value and dividing by the
# standard deviation of each RGB value; we've hardcoded the mean and std.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test); Datasets load
# training examples one at a time, so we wrap each Dataset in a DataLoader which
# iterates through the Dataset and forms minibatches. We divide the CIFAR-10
# training set into train and val sets by passing a Sampler object to the
# DataLoader telling how it should sample from the underlying Dataset.
cifar10_train = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./cs231n/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./cs231n/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


# Part II. Barebones PyTorch

PyTorch ships with high-level APIs to help us define model architectures conveniently, which we will cover in Part II of this tutorial. In this section, we will start with the barebone PyTorch elements to understand the autograd engine better. After this exercise, you will come to appreciate the high-level model API more.

We will start with a simple fully-connected ReLU network with two hidden layers and no biases for CIFAR classification.
This implementation computes the forward pass using operations on PyTorch Tensors, and uses PyTorch autograd to compute gradients. It is important that you understand every line, because you will write a harder version after the example.

When we create a PyTorch Tensor with `requires_grad=True`, then operations involving that Tensor will not just compute values; they will also build up a computational graph in the background, allowing us to easily backpropagate through the graph to compute gradients of some Tensors with respect to a downstream loss. Concretely if x is a Tensor with `x.requires_grad == True` then after backpropagation `x.grad` will be another Tensor holding the gradient of x with respect to the scalar loss at the end.

### PyTorch Tensors: Flatten Function
A PyTorch Tensor is conceptionally similar to a numpy array: it is an n-dimensional grid of numbers, and like numpy PyTorch provides many functions to efficiently operate on Tensors. As a simple example, we provide a `flatten` function below which reshapes image data for use in a fully-connected neural network.

Recall that image data is typically stored in a Tensor of shape N x C x H x W, where:

* N is the number of datapoints
* C is the number of channels
* H is the height of the intermediate feature map in pixels
* W is the height of the intermediate feature map in pixels

This is the right way to represent the data when we are doing something like a 2D convolution, that needs spatial understanding of where the intermediate features are relative to each other. When we use fully connected affine layers to process the image, however, we want each datapoint to be represented by a single vector -- it's no longer useful to segregate the different channels, rows, and columns of the data. So, we use a "flatten" operation to collapse the `C x H x W` values per representation into a single long vector. The flatten function below first reads in the N, C, H, and W values from a given batch of data, and then returns a "view" of that data. "View" is analogous to numpy's "reshape" method: it reshapes x's dimensions to be N x ??, where ?? is allowed to be anything (in this case, it will be C x H x W, but we don't need to specify that explicitly).

In [ ]:
def flatten(x):
    N = x.shape[0] # read in N, C, H, W
    return x.view(N, -1)  # "flatten" the C * H * W values into a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

Before flattening:  tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]]],


        [[[ 6,  7],
          [ 8,  9],
          [10, 11]]]])
After flattening:  tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


### Barebones PyTorch: Two-Layer Network

Here we define a function `two_layer_fc` which performs the forward pass of a two-layer fully-connected ReLU network on a batch of image data. After defining the forward pass we check that it doesn't crash and that it produces outputs of the right shape by running zeros through the network.

You don't have to write any code here, but it's important that you read and understand the implementation.

In [ ]:
import torch.nn.functional as F  # useful stateless functions

def two_layer_fc(x, params):
    """
    A fully-connected neural networks; the architecture is:
    NN is fully connected -> ReLU -> fully connected layer.
    Note that this function only defines the forward pass;
    PyTorch will take care of the backward pass for us.

    The input to the network will be a minibatch of data, of shape
    (N, d1, ..., dM) where d1 * ... * dM = D. The hidden layer will have H units,
    and the output layer will produce scores for C classes.

    Inputs:
    - x: A PyTorch Tensor of shape (N, d1, ..., dM) giving a minibatch of
      input data.
    - params: A list [w1, w2] of PyTorch Tensors giving weights for the network;
      w1 has shape (D, H) and w2 has shape (H, C).

    Returns:
    - scores: A PyTorch Tensor of shape (N, C) giving classification scores for
      the input data x.
    """
    # first we flatten the image
    x = flatten(x)  # shape: [batch_size, C x H x W]

    w1, w2 = params

    # Forward pass: compute predicted y using operations on Tensors. Since w1 and
    # w2 have requires_grad=True, operations involving these Tensors will cause
    # PyTorch to build a computational graph, allowing automatic computation of
    # gradients. Since we are no longer implementing the backward pass by hand we
    # don't need to keep references to intermediate values.
    # you can also use `.clamp(min=0)`, equivalent to F.relu()
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, feature dimension 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # you should see [64, 10]

two_layer_fc_test()

torch.Size([64, 10])


### Barebones PyTorch: Three-Layer ConvNet

Here you will complete the implementation of the function `three_layer_convnet`, which will perform the forward pass of a three-layer convolutional network. Like above, we can immediately test our implementation by passing zeros through the network. The network should have the following architecture:

1. A convolutional layer (with bias) with `channel_1` filters, each with shape `KW1 x KH1`, and zero-padding of two
2. ReLU nonlinearity
3. A convolutional layer (with bias) with `channel_2` filters, each with shape `KW2 x KH2`, and zero-padding of one
4. ReLU nonlinearity
5. Fully-connected layer with bias, producing scores for C classes.

Note that we have **no softmax activation** here after our fully-connected layer: this is because PyTorch's cross entropy loss performs a softmax activation for you, and by bundling that step in makes computation more efficient.

**HINT**: For convolutions: http://pytorch.org/docs/stable/nn.html#torch.nn.functional.conv2d; pay attention to the shapes of convolutional filters!

In [ ]:
def three_layer_convnet(x, params):
    """
    Performs the forward pass of a three-layer convolutional network with the
    architecture defined above.

    Inputs:
    - x: A PyTorch Tensor of shape (N, 3, H, W) giving a minibatch of images
    - params: A list of PyTorch Tensors giving the weights and biases for the
      network; should contain the following:
      - conv_w1: PyTorch Tensor of shape (channel_1, 3, KH1, KW1) giving weights
        for the first convolutional layer
      - conv_b1: PyTorch Tensor of shape (channel_1,) giving biases for the first
        convolutional layer
      - conv_w2: PyTorch Tensor of shape (channel_2, channel_1, KH2, KW2) giving
        weights for the second convolutional layer
      - conv_b2: PyTorch Tensor of shape (channel_2,) giving biases for the second
        convolutional layer
      - fc_w: PyTorch Tensor giving weights for the fully-connected layer. Can you
        figure out what the shape should be?
      - fc_b: PyTorch Tensor giving biases for the fully-connected layer. Can you
        figure out what the shape should be?

    Returns:
    - scores: PyTorch Tensor of shape (N, C) giving classification scores for x
    """
    conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b = params
    scores = None
    ################################################################################
    # TODO: Implement the forward pass for the three-layer ConvNet.                #
    ################################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    conv_out1 = F.conv2d(x, conv_w1, conv_b1, padding = 2)
    conv_out1_relu = F.relu(conv_out1)
    conv_out2 = F.conv2d(conv_out1_relu, conv_w2, conv_b2, padding = 1)
    conv_out2_relu = F.relu(conv_out2)

    conv_out2_relu = flatten(conv_out2_relu)
    scores = conv_out2_relu.mm(fc_w) + fc_b

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ################################################################################
    #                                 END OF YOUR CODE                             #
    ################################################################################
    return scores

After defining the forward pass of the ConvNet above, run the following cell to test your implementation.

When you run this function, scores should have shape (64, 10).

In [ ]:
def three_layer_convnet_test():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, image size [3, 32, 32]

    conv_w1 = torch.zeros((6, 3, 5, 5), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b1 = torch.zeros((6,))  # out_channel
    conv_w2 = torch.zeros((9, 6, 3, 3), dtype=dtype)  # [out_channel, in_channel, kernel_H, kernel_W]
    conv_b2 = torch.zeros((9,))  # out_channel

    # you must calculate the shape of the tensor after two conv layers, before the fully-connected layer
    fc_w = torch.zeros((9 * 32 * 32, 10))
    fc_b = torch.zeros(10)

    scores = three_layer_convnet(x, [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b])
    print(scores.size())  # you should see [64, 10]
three_layer_convnet_test()

torch.Size([64, 10])


### Barebones PyTorch: Initialization
Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852

In [ ]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # conv weight [out_channel, in_channel, kH, kW]
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# you should see the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))

tensor([[-0.1900,  0.1724, -0.3097, -0.2127, -0.8664],
        [-0.0588, -0.3194, -0.5377,  0.5989,  0.7119],
        [ 1.1511,  0.9085, -0.4152, -0.6657, -1.4098]], device='cuda:0',
       requires_grad=True)

### Barebones PyTorch: Check Accuracy
When training the model we will use the following function to check the accuracy of our model on the training or validation sets.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.

In [ ]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

### BareBones PyTorch: Training Loop
We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.

In [ ]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(loader_train):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

### BareBones PyTorch: Train a Two-Layer Network
Now we are ready to run the training loop. We need to explicitly allocate tensors for the fully connected weights, `w1` and `w2`.

Each minibatch of CIFAR has 64 examples, so the tensor shape is `[64, 3, 32, 32]`.

After flattening, `x` shape should be `[64, 3 * 32 * 32]`. This will be the size of the first dimension of `w1`.
The second dimension of `w1` is the hidden layer size, which will also be the first dimension of `w2`.

Finally, the output of the network is a 10-dimensional vector that represents the probability distribution over 10 classes.

You don't need to tune any hyperparameters but you should see accuracies above 40% after training for one epoch.

In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

Iteration 0, loss = 3.8420
Checking accuracy on the val set
Got 152 / 1000 correct (15.20%)

Iteration 100, loss = 2.2844
Checking accuracy on the val set
Got 327 / 1000 correct (32.70%)

Iteration 200, loss = 1.8962
Checking accuracy on the val set
Got 377 / 1000 correct (37.70%)

Iteration 300, loss = 1.9035
Checking accuracy on the val set
Got 378 / 1000 correct (37.80%)

Iteration 400, loss = 1.8229
Checking accuracy on the val set
Got 416 / 1000 correct (41.60%)

Iteration 500, loss = 1.9463
Checking accuracy on the val set
Got 391 / 1000 correct (39.10%)

Iteration 600, loss = 1.8252
Checking accuracy on the val set
Got 429 / 1000 correct (42.90%)

Iteration 700, loss = 1.7126
Checking accuracy on the val set
Got 429 / 1000 correct (42.90%)



### BareBones PyTorch: Training a ConvNet

In the below you should use the functions defined above to train a three-layer convolutional network on CIFAR. The network should have the following architecture:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

You should initialize your weight matrices using the `random_weight` function defined above, and you should initialize your bias vectors using the `zero_weight` function above.

You don't need to tune any hyperparameters, but if everything works correctly you should achieve an accuracy above 42% after one epoch.

In [ ]:
learning_rate = 3e-3

channel_1 = 32
channel_2 = 16

conv_w1 = None
conv_b1 = None
conv_w2 = None
conv_b2 = None
fc_w = None
fc_b = None

################################################################################
# TODO: Initialize the parameters of a three-layer ConvNet.                    #
################################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

H, W = (32, 32)
pad1, pad2 = (2,1)
HH1, WW1, HH2, WW2 = (5, 5, 3, 3)
stride = 1
num_classes = 10
H1_out = int(1 + (H + 2 * pad1 - HH1) / stride)
W1_out = int(1 + (W + 2 * pad1 - WW1) / stride)

H2_out = int(1 + (H1_out + 2 * pad2 - HH2) / stride)
W2_out = int(1 + (W1_out + 2 * pad2 - WW2) / stride)

conv_w1 = random_weight((channel_1,3,HH1,WW1))
conv_b1 = zero_weight((channel_1,))
conv_w2 = random_weight((channel_2,channel_1,HH2,WW2))
conv_b2 = zero_weight((channel_2,))

fc_w = random_weight((H2_out*W2_out*channel_2, num_classes))
fc_b = zero_weight((num_classes,))

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

params = [conv_w1, conv_b1, conv_w2, conv_b2, fc_w, fc_b]
train_part2(three_layer_convnet, params, learning_rate)

Iteration 0, loss = 2.9255
Checking accuracy on the val set
Got 111 / 1000 correct (11.10%)

Iteration 100, loss = 1.7427
Checking accuracy on the val set
Got 359 / 1000 correct (35.90%)

Iteration 200, loss = 1.4832
Checking accuracy on the val set
Got 395 / 1000 correct (39.50%)

Iteration 300, loss = 1.5917
Checking accuracy on the val set
Got 423 / 1000 correct (42.30%)

Iteration 400, loss = 1.7575
Checking accuracy on the val set
Got 452 / 1000 correct (45.20%)

Iteration 500, loss = 1.4731
Checking accuracy on the val set
Got 463 / 1000 correct (46.30%)

Iteration 600, loss = 1.3838
Checking accuracy on the val set
Got 475 / 1000 correct (47.50%)

Iteration 700, loss = 1.3786
Checking accuracy on the val set
Got 469 / 1000 correct (46.90%)



# Part III. PyTorch Module API

Barebone PyTorch requires that we track all the parameter tensors by hand. This is fine for small networks with a few tensors, but it would be extremely inconvenient and error-prone to track tens or hundreds of tensors in larger networks.

PyTorch provides the `nn.Module` API for you to define arbitrary network architectures, while tracking every learnable parameters for you. In Part II, we implemented SGD ourselves. PyTorch also provides the `torch.optim` package that implements all the common optimizers, such as RMSProp, Adagrad, and Adam. It even supports approximate second-order methods like L-BFGS! You can refer to the [doc](http://pytorch.org/docs/master/optim.html) for the exact specifications of each optimizer.

To use the Module API, follow the steps below:

1. Subclass `nn.Module`. Give your network class an intuitive name like `TwoLayerFC`.

2. In the constructor `__init__()`, define all the layers you need as class attributes. Layer objects like `nn.Linear` and `nn.Conv2d` are themselves `nn.Module` subclasses and contain learnable parameters, so that you don't have to instantiate the raw tensors yourself. `nn.Module` will track these internal parameters for you. Refer to the [doc](http://pytorch.org/docs/master/nn.html) to learn more about the dozens of builtin layers. **Warning**: don't forget to call the `super().__init__()` first!

3. In the `forward()` method, define the *connectivity* of your network. You should use the attributes defined in `__init__` as function calls that take tensor as input and output the "transformed" tensor. Do *not* create any new layers with learnable parameters in `forward()`! All of them must be declared upfront in `__init__`.

After you define your Module subclass, you can instantiate it as an object and call it just like the NN forward function in part II.

### Module API: Two-Layer Network
Here is a concrete example of a 2-layer fully connected network:

In [ ]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # assign layer objects to class attributes
        self.fc1 = nn.Linear(input_size, hidden_size)
        # nn.init package contains convenient initialization methods
        # http://pytorch.org/docs/master/nn.html#torch-nn-init
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        # forward always defines connectivity
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, feature dimension 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_TwoLayerFC()

torch.Size([64, 10])


### Module API: Three-Layer ConvNet
It's your turn to implement a 3-layer ConvNet followed by a fully connected layer. The network architecture should be the same as in Part II:

1. Convolutional layer with `channel_1` 5x5 filters with zero-padding of 2
2. ReLU
3. Convolutional layer with `channel_2` 3x3 filters with zero-padding of 1
4. ReLU
5. Fully-connected layer to `num_classes` classes

You should initialize the weight matrices of the model using the Kaiming normal initialization method.

**HINT**: http://pytorch.org/docs/stable/nn.html#conv2d

After you implement the three-layer ConvNet, the `test_ThreeLayerConvNet` function will run your implementation; it should print `(64, 10)` for the shape of the output scores.

In [ ]:
class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        ########################################################################
        # TODO: Set up the layers you need for a three-layer ConvNet with the  #
        # architecture defined above.                                          #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv2d1 = nn.Conv2d(in_channel, channel_1, kernel_size=(5,5), padding=2)
        nn.init.kaiming_normal_(self.conv2d1.weight, nonlinearity='relu')

        self.conv2d2 = nn.Conv2d(channel_1, channel_2, kernel_size=(3,3), padding=1)
        nn.init.kaiming_normal_(self.conv2d2.weight, nonlinearity='relu')

        self.fc = nn.Linear(in_features=channel_2*32*32, out_features=num_classes)
        nn.init.kaiming_normal_(self.fc.weight)


        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                          END OF YOUR CODE                            #
        ########################################################################

    def forward(self, x):
        scores = None
        ########################################################################
        # TODO: Implement the forward function for a 3-layer ConvNet. you      #
        # should use the layers you defined in __init__ and specify the        #
        # connectivity of those layers in forward()                            #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        conv_out1 = F.conv2d(x, self.conv2d1.weight, padding=2)
        conv_out1_relu = F.relu(conv_out1)
        conv_out2 = F.conv2d(conv_out1_relu, self.conv2d2.weight, padding = 1)
        conv_out2_relu = F.relu(conv_out2)

        conv_out2_relu = flatten(conv_out2_relu)
        scores = self.fc(conv_out2_relu)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                             END OF YOUR CODE                         #
        ########################################################################
        return scores


def test_ThreeLayerConvNet():
    x = torch.zeros((64, 3, 32, 32), dtype=dtype)  # minibatch size 64, image size [3, 32, 32]
    model = ThreeLayerConvNet(in_channel=3, channel_1=12, channel_2=8, num_classes=10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_ThreeLayerConvNet()

torch.Size([64, 10])


### Module API: Check Accuracy
Given the validation or test set, we can check the classification accuracy of a neural network.

This version is slightly different from the one in part II. You don't manually pass in the parameters anymore.

In [ ]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))
        return acc

### Module API: Training Loop
We also use a slightly different training loop. Rather than updating the values of the weights ourselves, we use an Optimizer object from the `torch.optim` package, which abstract the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.

In [ ]:
def train_part34(model, optimizer, epochs=1):
    """
    Train a model on CIFAR-10 using the PyTorch Module API.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Nothing, but prints model accuracies during training.
    """
    model = model.to(device=device)  # move the model parameters to CPU/GPU
    for e in range(epochs):
        print('Epoch {epoch:.2f}'.format(epoch=e))
        for t, (x, y) in enumerate(loader_train):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each  parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part34(loader_val, model)
                print()

### Module API: Train a Two-Layer Network
Now we are ready to run the training loop. In contrast to part II, we don't explicitly allocate parameter tensors anymore.

Simply pass the input size, hidden layer size, and number of classes (i.e. output size) to the constructor of `TwoLayerFC`.

You also need to define an optimizer that tracks all the learnable parameters inside `TwoLayerFC`.

You don't need to tune any hyperparameters, but you should see model accuracies above 40% after training for one epoch.

In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

Epoch 0.00
Iteration 0, loss = 3.0108
Checking accuracy on validation set
Got 172 / 1000 correct (17.20)

Iteration 100, loss = 2.2594
Checking accuracy on validation set
Got 326 / 1000 correct (32.60)

Iteration 200, loss = 1.5856
Checking accuracy on validation set
Got 391 / 1000 correct (39.10)

Iteration 300, loss = 1.8052
Checking accuracy on validation set
Got 371 / 1000 correct (37.10)

Iteration 400, loss = 1.9451
Checking accuracy on validation set
Got 388 / 1000 correct (38.80)

Iteration 500, loss = 1.8069
Checking accuracy on validation set
Got 422 / 1000 correct (42.20)

Iteration 600, loss = 1.4978
Checking accuracy on validation set
Got 439 / 1000 correct (43.90)

Iteration 700, loss = 1.9352
Checking accuracy on validation set
Got 468 / 1000 correct (46.80)



### Module API: Train a Three-Layer ConvNet
You should now use the Module API to train a three-layer ConvNet on CIFAR. This should look very similar to training the two-layer network! You don't need to tune any hyperparameters, but you should achieve above above 45% after training for one epoch.

You should train the model using stochastic gradient descent without momentum.

In [ ]:
learning_rate = 3e-3
channel_1 = 32
channel_2 = 16

model = None
optimizer = None
################################################################################
# TODO: Instantiate your ThreeLayerConvNet model and a corresponding optimizer #
################################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

model = ThreeLayerConvNet(3, channel_1, channel_2, 10)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

train_part34(model, optimizer)

Epoch 0.00
Iteration 0, loss = 2.8391
Checking accuracy on validation set
Got 113 / 1000 correct (11.30)

Iteration 100, loss = 2.0624
Checking accuracy on validation set
Got 299 / 1000 correct (29.90)

Iteration 200, loss = 1.9349
Checking accuracy on validation set
Got 298 / 1000 correct (29.80)

Iteration 300, loss = 1.9396
Checking accuracy on validation set
Got 352 / 1000 correct (35.20)

Iteration 400, loss = 1.8914
Checking accuracy on validation set
Got 375 / 1000 correct (37.50)

Iteration 500, loss = 1.6300
Checking accuracy on validation set
Got 411 / 1000 correct (41.10)

Iteration 600, loss = 1.8326
Checking accuracy on validation set
Got 432 / 1000 correct (43.20)

Iteration 700, loss = 1.3055
Checking accuracy on validation set
Got 472 / 1000 correct (47.20)



# Part IV. PyTorch Sequential API

Part III introduced the PyTorch Module API, which allows you to define arbitrary learnable layers and their connectivity.

For simple models like a stack of feed forward layers, you still need to go through 3 steps: subclass `nn.Module`, assign layers to class attributes in `__init__`, and call each layer one by one in `forward()`. Is there a more convenient way?

Fortunately, PyTorch provides a container Module called `nn.Sequential`, which merges the above steps into one. It is not as flexible as `nn.Module`, because you cannot specify more complex topology than a feed-forward stack, but it's good enough for many use cases.

### Sequential API: Two-Layer Network
Let's see how to rewrite our two-layer fully connected network example with `nn.Sequential`, and train it using the training loop defined above.

Again, you don't need to tune any hyperparameters here, but you shoud achieve above 40% accuracy after one epoch of training.

In [ ]:
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

Epoch 0.00
Iteration 0, loss = 2.3118
Checking accuracy on validation set
Got 132 / 1000 correct (13.20)

Iteration 100, loss = 2.0332
Checking accuracy on validation set
Got 366 / 1000 correct (36.60)

Iteration 200, loss = 2.1952
Checking accuracy on validation set
Got 388 / 1000 correct (38.80)

Iteration 300, loss = 1.8258
Checking accuracy on validation set
Got 433 / 1000 correct (43.30)

Iteration 400, loss = 1.6358
Checking accuracy on validation set
Got 438 / 1000 correct (43.80)

Iteration 500, loss = 1.7931
Checking accuracy on validation set
Got 445 / 1000 correct (44.50)

Iteration 600, loss = 1.5900
Checking accuracy on validation set
Got 453 / 1000 correct (45.30)

Iteration 700, loss = 1.8068
Checking accuracy on validation set
Got 453 / 1000 correct (45.30)



### Sequential API: Three-Layer ConvNet
Here you should use `nn.Sequential` to define and train a three-layer ConvNet with the same architecture we used in Part III:

1. Convolutional layer (with bias) with 32 5x5 filters, with zero-padding of 2
2. ReLU
3. Convolutional layer (with bias) with 16 3x3 filters, with zero-padding of 1
4. ReLU
5. Fully-connected layer (with bias) to compute scores for 10 classes

You can use the default PyTorch weight initialization.

You should optimize your model using stochastic gradient descent with Nesterov momentum 0.9.

Again, you don't need to tune any hyperparameters but you should see accuracy above 55% after one epoch of training.

In [ ]:
channel_1 = 32
channel_2 = 16
learning_rate = 1e-2

model = None
optimizer = None

################################################################################
# TODO: Rewrite the 2-layer ConvNet with bias from Part III with the           #
# Sequential API.                                                              #
################################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

model = nn.Sequential(
    nn.Conv2d(3, channel_1, (5,5), padding=2),
    nn.ReLU(),
    nn.Conv2d(channel_1, channel_2, (3,3), padding=1),
    nn.ReLU(),
    Flatten(),
    nn.Linear(channel_2*32*32, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

train_part34(model, optimizer, epochs=1)

Epoch 0.00
Iteration 0, loss = 2.2957
Checking accuracy on validation set
Got 112 / 1000 correct (11.20)

Iteration 100, loss = 1.6771
Checking accuracy on validation set
Got 441 / 1000 correct (44.10)

Iteration 200, loss = 1.4574
Checking accuracy on validation set
Got 502 / 1000 correct (50.20)

Iteration 300, loss = 1.3305
Checking accuracy on validation set
Got 524 / 1000 correct (52.40)

Iteration 400, loss = 1.5011
Checking accuracy on validation set
Got 535 / 1000 correct (53.50)

Iteration 500, loss = 1.4213
Checking accuracy on validation set
Got 547 / 1000 correct (54.70)

Iteration 600, loss = 1.2786
Checking accuracy on validation set
Got 583 / 1000 correct (58.30)

Iteration 700, loss = 1.1279
Checking accuracy on validation set
Got 574 / 1000 correct (57.40)



# Part V. CIFAR-10 open-ended challenge

In this section, you can experiment with whatever ConvNet architecture you'd like on CIFAR-10.

Now it's your job to experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves **at least 70%** accuracy on the CIFAR-10 **validation** set within 10 epochs. You can use the check_accuracy and train functions from above. You can use either `nn.Module` or `nn.Sequential` API.

Describe what you did at the end of this notebook.

Here are the official API documentation for each component. One note: what we call in the class "spatial batch norm" is called "BatchNorm2D" in PyTorch.

* Layers in torch.nn package: http://pytorch.org/docs/stable/nn.html
* Activations: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* Loss functions: http://pytorch.org/docs/stable/nn.html#loss-functions
* Optimizers: http://pytorch.org/docs/stable/optim.html


### Things you might try:
- **Filter size**: Above we used 5x5; would smaller filters be more efficient?
- **Number of filters**: Above we used 32 filters. Do more or fewer do better?
- **Pooling vs Strided Convolution**: Do you use max pooling or just stride convolutions?
- **Batch normalization**: Try adding spatial batch normalization after convolution layers and vanilla batch normalization after affine layers. Do your networks train faster?
- **Network architecture**: The network above has two layers of trainable parameters. Can you do better with a deep network? Good architectures to try include:
    - [conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [conv-relu-conv-relu-pool]xN -> [affine]xM -> [softmax or SVM]
    - [batchnorm-relu-conv]xN -> [affine]xM -> [softmax or SVM]
- **Global Average Pooling**: Instead of flattening and then having multiple affine layers, perform convolutions until your image gets small (7x7 or so) and then perform an average pooling operation to get to a 1x1 image picture (1, 1 , Filter#), which is then reshaped into a (Filter#) vector. This is used in [Google's Inception Network](https://arxiv.org/abs/1512.00567) (See Table 1 for their architecture).
- **Regularization**: Add l2 weight regularization, or perhaps use Dropout.

### Tips for training
For each network architecture that you try, you should tune the learning rate and other hyperparameters. When doing this there are a couple important things to keep in mind:

- If the parameters are working well, you should see improvement within a few hundred iterations
- Remember the coarse-to-fine approach for hyperparameter tuning: start by testing a large range of hyperparameters for just a few training iterations to find the combinations of parameters that are working at all.
- Once you have found some sets of parameters that seem to work, search more finely around these parameters. You may need to train for more epochs.
- You should use the validation set for hyperparameter search, and save your test set for evaluating your architecture on the best parameters as selected by the validation set.

### Going above and beyond
If you are feeling adventurous there are many other features you can implement to try and improve your performance. You are **not required** to implement any of these, but don't miss the fun if you have time!

- Alternative optimizers: you can try Adam, Adagrad, RMSprop, etc.
- Alternative activation functions such as leaky ReLU, parametric ReLU, ELU, or MaxOut.
- Model ensembles
- Data augmentation
- New Architectures
  - [ResNets](https://arxiv.org/abs/1512.03385) where the input from the previous layer is added to the output.
  - [DenseNets](https://arxiv.org/abs/1608.06993) where inputs into previous layers are concatenated together.
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)

### Have fun and happy training!

In [ ]:
################################################################################
# TODO:                                                                        #
# Experiment with any architectures, optimizers, and hyperparameters.          #
# Achieve AT LEAST 70% accuracy on the *validation set* within 10 epochs.      #
#                                                                              #
# Note that you can use the check_accuracy function to evaluate on either      #
# the test set or the validation set, by passing either loader_test or         #
# loader_val as the second argument to check_accuracy. You should not touch    #
# the test set until you have finished your architecture and  hyperparameter   #
# tuning, and only run the test set once at the end to report a final value.   #
################################################################################
model = None
optimizer = None

# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

channel_1 = 32
channel_2 = 16
learning_rate = 1e-2
filter_1 = (3,3)
filter_2 = (3,3)

model = nn.Sequential(
    nn.Conv2d(3, channel_1, filter_1, padding=1),
    nn.ReLU(),
    nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
    nn.ReLU(),
    Flatten(),
    nn.Linear(channel_2*32*32, 10),
)

optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)


# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

# You should get at least 70% accuracy.
# You may modify the number of epochs to any number below 15.
# train_part34(model, optimizer, epochs=4)

In [ ]:
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: True
    weight_decay: 0
)

In [ ]:
channel_1_bounds = [16, 128]
channel_2_bounds= [8, 64]
learning_rate = 1e-3
filter_1 = (3,3)
filter_2 = (3,3)

models_by_channel = dict()

for i in range(8):
    channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    print('Channel 1 is {channel}'.format(channel=channel_1))
    print('Channel 2 is {channel}'.format(channel=channel_2))
    models_by_channel[(channel_1, channel_2)] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
)
    optimizer = optim.Adam(models_by_channel[(channel_1, channel_2)].parameters(), lr=learning_rate)

    train_part34(models_by_channel[(channel_1, channel_2)], optimizer, epochs=2)

Channel 1 is 82
Channel 2 is 15
Epoch 0.00
Iteration 0, loss = 2.3057
Checking accuracy on validation set
Got 131 / 1000 correct (13.10)

Iteration 100, loss = 1.4566
Checking accuracy on validation set
Got 449 / 1000 correct (44.90)

Iteration 200, loss = 1.2415
Checking accuracy on validation set
Got 526 / 1000 correct (52.60)

Iteration 300, loss = 1.3440
Checking accuracy on validation set
Got 567 / 1000 correct (56.70)

Iteration 400, loss = 1.3755
Checking accuracy on validation set
Got 576 / 1000 correct (57.60)

Iteration 500, loss = 1.0490
Checking accuracy on validation set
Got 581 / 1000 correct (58.10)

Iteration 600, loss = 1.1749
Checking accuracy on validation set
Got 603 / 1000 correct (60.30)

Iteration 700, loss = 1.0077
Checking accuracy on validation set
Got 612 / 1000 correct (61.20)

Epoch 1.00
Iteration 0, loss = 1.0058
Checking accuracy on validation set
Got 616 / 1000 correct (61.60)

Iteration 100, loss = 0.9630
Checking accuracy on validation set
Got 625 / 10

In [ ]:
for key, model in models_by_channel.items():
  print(key)
  check_accuracy_part34(loader_val, model)

(82, 15)
Checking accuracy on validation set
Got 640 / 1000 correct (64.00)
(32, 59)
Checking accuracy on validation set
Got 668 / 1000 correct (66.80)
(60, 19)
Checking accuracy on validation set
Got 657 / 1000 correct (65.70)
(94, 8)
Checking accuracy on validation set
Got 587 / 1000 correct (58.70)
(103, 59)
Checking accuracy on validation set
Got 638 / 1000 correct (63.80)
(62, 38)
Checking accuracy on validation set
Got 651 / 1000 correct (65.10)
(126, 26)
Checking accuracy on validation set
Got 655 / 1000 correct (65.50)
(69, 25)
Checking accuracy on validation set
Got 642 / 1000 correct (64.20)


In [ ]:
channel_1_bounds = [28, 40]
channel_2_bounds= [48, 60]
learning_rate = 1e-3
filter_1 = (3,3)
filter_2 = (3,3)

models_by_channel_2 = dict()

for i in range(8):
    channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    print('Channel 1 is {channel}'.format(channel=channel_1))
    print('Channel 2 is {channel}'.format(channel=channel_2))
    models_by_channel_2[(channel_1, channel_2)] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
)
    optimizer = optim.Adam(models_by_channel_2[(channel_1, channel_2)].parameters(), lr=learning_rate)

    train_part34(models_by_channel_2[(channel_1, channel_2)], optimizer, epochs=2)

Channel 1 is 32
Channel 2 is 56
Epoch 0.00
Iteration 0, loss = 2.3034
Checking accuracy on validation set
Got 107 / 1000 correct (10.70)

Iteration 100, loss = 1.4346
Checking accuracy on validation set
Got 488 / 1000 correct (48.80)

Iteration 200, loss = 1.1946
Checking accuracy on validation set
Got 546 / 1000 correct (54.60)

Iteration 300, loss = 0.9771
Checking accuracy on validation set
Got 569 / 1000 correct (56.90)

Iteration 400, loss = 1.0778
Checking accuracy on validation set
Got 596 / 1000 correct (59.60)

Iteration 500, loss = 1.2007
Checking accuracy on validation set
Got 624 / 1000 correct (62.40)

Iteration 600, loss = 1.2288
Checking accuracy on validation set
Got 653 / 1000 correct (65.30)

Iteration 700, loss = 0.9658
Checking accuracy on validation set
Got 644 / 1000 correct (64.40)

Epoch 1.00
Iteration 0, loss = 0.7485
Checking accuracy on validation set
Got 649 / 1000 correct (64.90)

Iteration 100, loss = 0.8424
Checking accuracy on validation set
Got 651 / 10

In [ ]:
for key, model in models_by_channel_2.items():
  print(key)
  check_accuracy_part34(loader_val, model)

(32, 56)
Checking accuracy on validation set
Got 662 / 1000 correct (66.20)
(39, 57)
Checking accuracy on validation set
Got 668 / 1000 correct (66.80)
(37, 52)
Checking accuracy on validation set
Got 666 / 1000 correct (66.60)
(30, 56)
Checking accuracy on validation set
Got 670 / 1000 correct (67.00)
(29, 53)
Checking accuracy on validation set
Got 678 / 1000 correct (67.80)
(31, 50)
Checking accuracy on validation set
Got 667 / 1000 correct (66.70)
(31, 54)
Checking accuracy on validation set
Got 683 / 1000 correct (68.30)
(30, 50)
Checking accuracy on validation set
Got 663 / 1000 correct (66.30)


In [ ]:
learning_rate = 1e-3
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
model_best_filter_batch = nn.Sequential(
    nn.Conv2d(3, channel_1, filter_1, padding=1),
    nn.BatchNorm2d(channel_1),
    nn.ReLU(),
    nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
    nn.BatchNorm2d(channel_2),
    nn.ReLU(),
    nn.MaxPool2d((2,2)),
    Flatten(),
    nn.Linear(channel_2*16*16, 10),
    nn.BatchNorm1d(10)
)

optimizer = optim.Adam(model_best_filter_batch.parameters(), lr=learning_rate)

train_part34(model_best_filter_batch, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 2.5620
Checking accuracy on validation set
Got 119 / 1000 correct (11.90)

Iteration 100, loss = 1.4759
Checking accuracy on validation set
Got 496 / 1000 correct (49.60)

Iteration 200, loss = 1.3545
Checking accuracy on validation set
Got 584 / 1000 correct (58.40)

Iteration 300, loss = 1.3827
Checking accuracy on validation set
Got 614 / 1000 correct (61.40)

Iteration 400, loss = 1.0703
Checking accuracy on validation set
Got 635 / 1000 correct (63.50)

Iteration 500, loss = 1.1037
Checking accuracy on validation set
Got 594 / 1000 correct (59.40)

Iteration 600, loss = 1.3148
Checking accuracy on validation set
Got 635 / 1000 correct (63.50)

Iteration 700, loss = 1.0643
Checking accuracy on validation set
Got 640 / 1000 correct (64.00)

Epoch 1.00
Iteration 0, loss = 1.0776
Checking accuracy on validation set
Got 653 / 1000 correct (65.30)

Iteration 100, loss = 1.1357
Checking accuracy on validation set
Got 650 / 1000 correct (65.00)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model_best_filter_batch)

Checking accuracy on validation set
Got 659 / 1000 correct (65.90)


0.659

In [ ]:
lr_pow_bounds = [-3, -7]
reg_pow_bounds = [-2, -8]
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
models_lr_reg = dict()

for i in range(8):
    lr = 10**np.random.uniform(lr_pow_bounds[1], lr_pow_bounds[0])
    reg = 10**np.random.uniform(reg_pow_bounds[1], reg_pow_bounds[0])
    print('lr is {lr}'.format(lr=lr))
    print('reg is {reg}'.format(reg=reg))
    models_lr_reg[(lr, reg)] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
)
    optimizer = optim.Adam(models_lr_reg[(lr, reg)].parameters(), lr=lr, weight_decay=reg)

    train_part34(models_lr_reg[(lr, reg)], optimizer, epochs=5)

lr is 2.4779116379347956e-07
reg is 1.2736103059289747e-06
Epoch 0.00
Iteration 0, loss = 2.3280
Checking accuracy on validation set
Got 111 / 1000 correct (11.10)

Iteration 100, loss = 2.2970
Checking accuracy on validation set
Got 114 / 1000 correct (11.40)

Iteration 200, loss = 2.2841
Checking accuracy on validation set
Got 129 / 1000 correct (12.90)

Iteration 300, loss = 2.2901
Checking accuracy on validation set
Got 139 / 1000 correct (13.90)

Iteration 400, loss = 2.2759
Checking accuracy on validation set
Got 151 / 1000 correct (15.10)

Iteration 500, loss = 2.2924
Checking accuracy on validation set
Got 163 / 1000 correct (16.30)

Iteration 600, loss = 2.2637
Checking accuracy on validation set
Got 164 / 1000 correct (16.40)

Iteration 700, loss = 2.2914
Checking accuracy on validation set
Got 169 / 1000 correct (16.90)

Epoch 1.00
Iteration 0, loss = 2.2624
Checking accuracy on validation set
Got 168 / 1000 correct (16.80)

Iteration 100, loss = 2.2283
Checking accuracy on 

In [ ]:
for key, model in models_lr_reg.items():
  print(key)
  check_accuracy_part34(loader_val, model)

(2.4779116379347956e-07, 1.2736103059289747e-06)
Checking accuracy on validation set
Got 277 / 1000 correct (27.70)
(1.8120346140707384e-05, 1.4649092022585413e-05)
Checking accuracy on validation set
Got 529 / 1000 correct (52.90)
(3.4213029311418195e-06, 4.3750628632570215e-07)
Checking accuracy on validation set
Got 444 / 1000 correct (44.40)
(0.00011166294562595948, 0.001025423660548774)
Checking accuracy on validation set
Got 644 / 1000 correct (64.40)
(0.0008507429273740494, 0.00040079006432613816)
Checking accuracy on validation set
Got 700 / 1000 correct (70.00)
(0.0009333181049056624, 6.105751645328961e-08)
Checking accuracy on validation set
Got 697 / 1000 correct (69.70)
(6.624418173630821e-07, 1.217799702655641e-06)
Checking accuracy on validation set
Got 337 / 1000 correct (33.70)
(4.33218735749982e-05, 2.5309904101963763e-06)
Checking accuracy on validation set
Got 578 / 1000 correct (57.80)


In [ ]:
lr_bounds = [7e-4, 9.9e-4]
reg_pow_bounds = [-3, -8]
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
models_lr_reg_2 = dict()

for i in range(8):
    lr = np.random.uniform(lr_bounds[0], lr_bounds[1])
    reg = 10**np.random.uniform(reg_pow_bounds[1], reg_pow_bounds[0])
    print('lr is {lr}'.format(lr=lr))
    print('reg is {reg}'.format(reg=reg))
    models_lr_reg_2[(lr, reg)] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
)
    optimizer = optim.Adam(models_lr_reg_2[(lr, reg)].parameters(), lr=lr, weight_decay=reg)

    train_part34(models_lr_reg_2[(lr, reg)], optimizer, epochs=10)

lr is 0.0009219215516202864
reg is 1.966349236159055e-07
Epoch 0.00
Iteration 0, loss = 2.3221
Checking accuracy on validation set
Got 123 / 1000 correct (12.30)

Iteration 100, loss = 1.4994
Checking accuracy on validation set
Got 480 / 1000 correct (48.00)

Iteration 200, loss = 1.5986
Checking accuracy on validation set
Got 534 / 1000 correct (53.40)

Iteration 300, loss = 1.1796
Checking accuracy on validation set
Got 569 / 1000 correct (56.90)

Iteration 400, loss = 1.2599
Checking accuracy on validation set
Got 589 / 1000 correct (58.90)

Iteration 500, loss = 1.1595
Checking accuracy on validation set
Got 609 / 1000 correct (60.90)

Iteration 600, loss = 1.1346
Checking accuracy on validation set
Got 623 / 1000 correct (62.30)

Iteration 700, loss = 0.9853
Checking accuracy on validation set
Got 631 / 1000 correct (63.10)

Epoch 1.00
Iteration 0, loss = 0.9325
Checking accuracy on validation set
Got 649 / 1000 correct (64.90)

Iteration 100, loss = 0.8054
Checking accuracy on va

In [ ]:
for key, model in models_lr_reg_2.items():
  print(key)
  check_accuracy_part34(loader_val, model)

(0.0009219215516202864, 1.966349236159055e-07)
Checking accuracy on validation set
Got 651 / 1000 correct (65.10)
(0.0008023716805682793, 1.0190843996171629e-05)
Checking accuracy on validation set
Got 667 / 1000 correct (66.70)
(0.000872138477053398, 3.9692269785432135e-06)
Checking accuracy on validation set
Got 668 / 1000 correct (66.80)
(0.000854626515823378, 1.7072689456325607e-06)
Checking accuracy on validation set
Got 666 / 1000 correct (66.60)
(0.0008672952228293737, 5.908468444995261e-05)
Checking accuracy on validation set
Got 648 / 1000 correct (64.80)
(0.0009820412793869193, 4.889592391907814e-07)
Checking accuracy on validation set
Got 679 / 1000 correct (67.90)
(0.0008221459880592824, 2.5804887469366918e-05)
Checking accuracy on validation set
Got 663 / 1000 correct (66.30)
(0.0008902995733797961, 0.0006163390996146715)
Checking accuracy on validation set
Got 660 / 1000 correct (66.00)


In [ ]:
lr = 8.5e-4
reg = 0
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
model_lr_reg_dropout = dict()

model_lr_reg_dropout = nn.Sequential(
  nn.Conv2d(3, channel_1, filter_1, padding=1),
  nn.BatchNorm2d(channel_1),
  nn.Dropout2d(),
  nn.ReLU(),
  nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
  nn.BatchNorm2d(channel_2),
  nn.Dropout2d(),
  nn.ReLU(),
  nn.MaxPool2d((2,2)),
  Flatten(),
  nn.Linear(channel_2*16*16, 10),
  nn.BatchNorm1d(10),
  nn.Dropout()
)

optimizer = optim.Adam(model_lr_reg_dropout.parameters(), lr=lr, weight_decay=reg)

train_part34(model_lr_reg_dropout, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 3.4031
Checking accuracy on validation set
Got 124 / 1000 correct (12.40)

Iteration 100, loss = 2.4159
Checking accuracy on validation set
Got 324 / 1000 correct (32.40)

Iteration 200, loss = 2.0172
Checking accuracy on validation set
Got 392 / 1000 correct (39.20)

Iteration 300, loss = 2.2492
Checking accuracy on validation set
Got 403 / 1000 correct (40.30)

Iteration 400, loss = 1.8800
Checking accuracy on validation set
Got 464 / 1000 correct (46.40)

Iteration 500, loss = 2.1901
Checking accuracy on validation set
Got 479 / 1000 correct (47.90)

Iteration 600, loss = 1.8996
Checking accuracy on validation set
Got 481 / 1000 correct (48.10)

Iteration 700, loss = 1.9534
Checking accuracy on validation set
Got 464 / 1000 correct (46.40)

Epoch 1.00
Iteration 0, loss = 2.0796
Checking accuracy on validation set
Got 488 / 1000 correct (48.80)

Iteration 100, loss = 1.9842
Checking accuracy on validation set
Got 512 / 1000 correct (51.20)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model_lr_reg_dropout)

Checking accuracy on validation set
Got 625 / 1000 correct (62.50)


In [ ]:
lr = 8.5e-4
reg = 0
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
model_lr_reg_dropout = dict()

model_lr_reg_dropout = nn.Sequential(
  nn.Conv2d(3, channel_1, filter_1, padding=1),
  nn.BatchNorm2d(channel_1),
  nn.Dropout2d(0.3),
  nn.ReLU(),
  nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
  nn.BatchNorm2d(channel_2),
  nn.Dropout2d(0.3),
  nn.ReLU(),
  nn.MaxPool2d((2,2)),
  Flatten(),
  nn.Linear(channel_2*16*16, 10),
  nn.BatchNorm1d(10),
  nn.Dropout(0.3)
)

optimizer = optim.Adam(model_lr_reg_dropout.parameters(), lr=lr, weight_decay=reg)

train_part34(model_lr_reg_dropout, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 3.0919
Checking accuracy on validation set
Got 170 / 1000 correct (17.00)

Iteration 100, loss = 1.8075
Checking accuracy on validation set
Got 419 / 1000 correct (41.90)

Iteration 200, loss = 1.8957
Checking accuracy on validation set
Got 481 / 1000 correct (48.10)

Iteration 300, loss = 1.8655
Checking accuracy on validation set
Got 502 / 1000 correct (50.20)

Iteration 400, loss = 1.6727
Checking accuracy on validation set
Got 516 / 1000 correct (51.60)

Iteration 500, loss = 1.7622
Checking accuracy on validation set
Got 537 / 1000 correct (53.70)

Iteration 600, loss = 1.5942
Checking accuracy on validation set
Got 542 / 1000 correct (54.20)

Iteration 700, loss = 1.8885
Checking accuracy on validation set
Got 572 / 1000 correct (57.20)

Epoch 1.00
Iteration 0, loss = 1.5803
Checking accuracy on validation set
Got 548 / 1000 correct (54.80)

Iteration 100, loss = 1.6247
Checking accuracy on validation set
Got 579 / 1000 correct (57.90)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model_lr_reg_dropout)

Checking accuracy on validation set
Got 669 / 1000 correct (66.90)


In [ ]:
lr = 8.5e-4
reg = 0
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 28
channel_2 = 56
model_lr_reg_dropout = dict()

model_lr_reg_dropout = nn.Sequential(
  nn.Conv2d(3, channel_1, filter_1, padding=1),
  nn.BatchNorm2d(channel_1),
  nn.Dropout2d(0.1),
  nn.ReLU(),
  nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
  nn.BatchNorm2d(channel_2),
  nn.Dropout2d(0.1),
  nn.ReLU(),
  nn.MaxPool2d((2,2)),
  Flatten(),
  nn.Linear(channel_2*16*16, 10),
  nn.BatchNorm1d(10),
  nn.Dropout(0.3)
)

optimizer = optim.Adam(model_lr_reg_dropout.parameters(), lr=lr, weight_decay=reg)

train_part34(model_lr_reg_dropout, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 2.8314
Checking accuracy on validation set
Got 174 / 1000 correct (17.40)

Iteration 100, loss = 1.9416
Checking accuracy on validation set
Got 457 / 1000 correct (45.70)

Iteration 200, loss = 1.7402
Checking accuracy on validation set
Got 529 / 1000 correct (52.90)

Iteration 300, loss = 1.7353
Checking accuracy on validation set
Got 527 / 1000 correct (52.70)

Iteration 400, loss = 1.6396
Checking accuracy on validation set
Got 578 / 1000 correct (57.80)

Iteration 500, loss = 1.6376
Checking accuracy on validation set
Got 586 / 1000 correct (58.60)

Iteration 600, loss = 1.3578
Checking accuracy on validation set
Got 585 / 1000 correct (58.50)

Iteration 700, loss = 1.5409
Checking accuracy on validation set
Got 618 / 1000 correct (61.80)

Epoch 1.00
Iteration 0, loss = 1.5902
Checking accuracy on validation set
Got 628 / 1000 correct (62.80)

Iteration 100, loss = 1.5242
Checking accuracy on validation set
Got 590 / 1000 correct (59.00)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model_lr_reg_dropout)

Checking accuracy on validation set
Got 690 / 1000 correct (69.00)


In [ ]:
filter_1 = (3,3)
filter_2 = (3,3)
lr_pow_bounds = [-4, -2]
channel_1_bounds = [24,96]
channel_2_bounds = [12,64]
dropout2d_bounds = [0,0.5]
dropout1d_bounds = [0,0.5]
models_all = dict()

for i in range(32):
    lr = 10**np.random.uniform(lr_pow_bounds[0], lr_pow_bounds[1])
    channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    dropout2d_p = np.random.uniform(dropout2d_bounds[0], dropout2d_bounds[1])
    dropout1d_p = np.random.uniform(dropout1d_bounds[0], dropout2d_bounds[1])
    print('========================================================================')
    print('lr is {lr}'.format(lr=lr))
    print('channel_1 is {channel}'.format(channel=channel_1))
    print('channel_2 is {channel}'.format(channel=channel_2))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=dropout2d_p))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=dropout1d_p))
    print('========================================================================')
    index = (lr, channel_1, channel_2, dropout2d_p, dropout1d_p)
    models_all[index] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
      nn.Dropout1d(dropout1d_p)
)
    optimizer = optim.Adam(models_all[index].parameters(), lr=lr)

    train_part34(models_all[index], optimizer, epochs=4)

lr is 0.0032842655848992494
channel_1 is 30
channel_2 is 48
dropout2d_p is 0.055735244857738175
dropout1d_p is 0.08461718892630526
Epoch 0.00
Iteration 0, loss = 2.5557
Checking accuracy on validation set
Got 119 / 1000 correct (11.90)

Iteration 100, loss = 2.1833
Checking accuracy on validation set
Got 295 / 1000 correct (29.50)

Iteration 200, loss = 2.1404
Checking accuracy on validation set
Got 369 / 1000 correct (36.90)

Iteration 300, loss = 2.0755
Checking accuracy on validation set
Got 426 / 1000 correct (42.60)

Iteration 400, loss = 1.7785
Checking accuracy on validation set
Got 466 / 1000 correct (46.60)

Iteration 500, loss = 1.3859
Checking accuracy on validation set
Got 495 / 1000 correct (49.50)

Iteration 600, loss = 1.8489
Checking accuracy on validation set
Got 478 / 1000 correct (47.80)

Iteration 700, loss = 1.4781
Checking accuracy on validation set
Got 520 / 1000 correct (52.00)

Epoch 1.00
Iteration 0, loss = 1.4566
Checking accuracy on validation set
Got 525 / 

In [ ]:
for key, model in models_all.items():
    print('========================================================================')
    print('lr is {lr}'.format(lr=key[0]))
    print('channel_1 is {channel}'.format(channel=key[1]))
    print('channel_2 is {channel}'.format(channel=key[2]))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=key[3]))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=key[4]))
    print('========================================================================')
    check_accuracy_part34(loader_val, model)

lr is 0.0032842655848992494
channel_1 is 30
channel_2 is 48
dropout2d_p is 0.055735244857738175
dropout1d_p is 0.08461718892630526
Checking accuracy on validation set
Got 670 / 1000 correct (67.00)
lr is 0.0007044793590981516
channel_1 is 88
channel_2 is 14
dropout2d_p is 0.16540245044372165
dropout1d_p is 0.4397028220093211
Checking accuracy on validation set
Got 635 / 1000 correct (63.50)
lr is 0.009708730261970944
channel_1 is 40
channel_2 is 35
dropout2d_p is 0.3384934340415513
dropout1d_p is 0.13979412038546302
Checking accuracy on validation set
Got 531 / 1000 correct (53.10)
lr is 0.0004680742049498807
channel_1 is 57
channel_2 is 20
dropout2d_p is 0.15666711995441562
dropout1d_p is 0.0361639396010906
Checking accuracy on validation set
Got 656 / 1000 correct (65.60)
lr is 0.0008149315910215245
channel_1 is 41
channel_2 is 22
dropout2d_p is 0.29940841404073604
dropout1d_p is 0.0321338011893923
Checking accuracy on validation set
Got 632 / 1000 correct (63.20)
lr is 0.00176622116

In [ ]:
print_every = 240

filter_1 = (3,3)
filter_2 = (3,3)
lr_bounds = [1e-3, 2e-3]
channel_1_bounds = [30, 35]
channel_2_bounds = [44, 50]
dropout2d_bounds = [0.07,0.09]
dropout1d_bounds = [0.08,0.12]
models_all_2 = dict()

for i in range(10):
    lr = np.random.uniform(lr_bounds[0], lr_bounds[1])
    channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    dropout2d_p = np.random.uniform(dropout2d_bounds[0], dropout2d_bounds[1])
    dropout1d_p = np.random.uniform(dropout1d_bounds[0], dropout2d_bounds[1])
    print('========================================================================')
    print('COMBINATION {i}'.format(i=i))
    print('lr is {lr}'.format(lr=lr))
    print('channel_1 is {channel}'.format(channel=channel_1))
    print('channel_2 is {channel}'.format(channel=channel_2))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=dropout2d_p))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=dropout1d_p))
    print('========================================================================')
    index = (lr, channel_1, channel_2, dropout2d_p, dropout1d_p)
    models_all_2[index] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 10),
      nn.Dropout1d(dropout1d_p)
)
    optimizer = optim.Adam(models_all_2[index].parameters(), lr=lr)

    train_part34(models_all_2[index], optimizer, epochs=10)

COMBINATION 0
lr is 0.0017196110188349761
channel_1 is 33
channel_2 is 44
dropout2d_p is 0.08028761520160058
dropout1d_p is 0.08943016267411819
Epoch 0.00
Iteration 0, loss = 2.3378
Checking accuracy on validation set
Got 120 / 1000 correct (12.00)

Iteration 240, loss = 1.5356
Checking accuracy on validation set
Got 431 / 1000 correct (43.10)

Iteration 480, loss = 1.4929
Checking accuracy on validation set
Got 494 / 1000 correct (49.40)

Iteration 720, loss = 1.3193
Checking accuracy on validation set
Got 528 / 1000 correct (52.80)

Epoch 1.00
Iteration 0, loss = 1.5509
Checking accuracy on validation set
Got 560 / 1000 correct (56.00)

Iteration 240, loss = 1.4383
Checking accuracy on validation set
Got 590 / 1000 correct (59.00)

Iteration 480, loss = 1.2210
Checking accuracy on validation set
Got 598 / 1000 correct (59.80)

Iteration 720, loss = 1.2419
Checking accuracy on validation set
Got 609 / 1000 correct (60.90)

Epoch 2.00
Iteration 0, loss = 1.2992
Checking accuracy on val

In [ ]:
for key, model in models_all_2.items():
    print('========================================================================')
    if check_accuracy_part34(loader_val, model)>=0.7:
      best_model_1 = model
    print('lr is {lr}'.format(lr=key[0]))
    print('channel_1 is {channel}'.format(channel=key[1]))
    print('channel_2 is {channel}'.format(channel=key[2]))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=key[3]))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=key[4]))
    print('========================================================================')


Checking accuracy on validation set
Got 680 / 1000 correct (68.00)
lr is 0.0017196110188349761
channel_1 is 33
channel_2 is 44
dropout2d_p is 0.08028761520160058
dropout1d_p is 0.08943016267411819
Checking accuracy on validation set
Got 681 / 1000 correct (68.10)
lr is 0.0014233420599226638
channel_1 is 33
channel_2 is 49
dropout2d_p is 0.08359343612762186
dropout1d_p is 0.08058570343733905
Checking accuracy on validation set
Got 678 / 1000 correct (67.80)
lr is 0.0011882620204440256
channel_1 is 33
channel_2 is 45
dropout2d_p is 0.08620333737059906
dropout1d_p is 0.0825020270606992
Checking accuracy on validation set
Got 693 / 1000 correct (69.30)
lr is 0.0016040305832855893
channel_1 is 31
channel_2 is 49
dropout2d_p is 0.08926100261526063
dropout1d_p is 0.08979472461596805
Checking accuracy on validation set
Got 690 / 1000 correct (69.00)
lr is 0.0015610565119679415
channel_1 is 30
channel_2 is 47
dropout2d_p is 0.08020625169054696
dropout1d_p is 0.08117477920760832
Checking accurac

In [ ]:
best_model_1

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): Dropout2d(p=0.07890725779151218, inplace=False)
  (3): ReLU()
  (4): Conv2d(32, 44, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(44, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): Dropout2d(p=0.07890725779151218, inplace=False)
  (7): ReLU()
  (8): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (9): Flatten()
  (10): Linear(in_features=11264, out_features=10, bias=True)
  (11): Dropout1d(p=0.08201849592829381, inplace=False)
)

In [ ]:
PATH = '/content/drive/My Drive/{}models/best_model_2'.format(FOLDERNAME)
torch.save(best_model_1, PATH)

In [ ]:
the_model = torch.load(PATH)

In [ ]:
the_model

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): Dropout2d(p=0.07890725779151218, inplace=False)
  (3): ReLU()
  (4): Conv2d(32, 44, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(44, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): Dropout2d(p=0.07890725779151218, inplace=False)
  (7): ReLU()
  (8): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (9): Flatten()
  (10): Linear(in_features=11264, out_features=10, bias=True)
  (11): Dropout1d(p=0.08201849592829381, inplace=False)
)

In [ ]:
print_every = 240

filter_1 = (3,3)
filter_2 = (3,3)
lr_bounds = [1e-3, 5e-3]
# channel_1_bounds = [30, 35]
# channel_2_bounds = [44, 50]
dropout2d_bounds = [0,0.1]
dropout1d_bounds = [0,0.1]
# lr = 1e-3
channel_1 = 36
channel_2 = 48
# dropout2d_bounds = 0.08
# dropout1d_bounds = 0.1
models_conv2_linear2 = dict()

for i in range(10):
    lr = np.random.uniform(lr_bounds[0], lr_bounds[1])
    # channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    # channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    dropout2d_p = np.random.uniform(dropout2d_bounds[0], dropout2d_bounds[1])
    dropout1d_p = np.random.uniform(dropout1d_bounds[0], dropout2d_bounds[1])
    print('========================================================================')
    print('COMBINATION {i}'.format(i=i))
    print('lr is {lr}'.format(lr=lr))
    # print('channel_1 is {channel}'.format(channel=channel_1))
    # print('channel_2 is {channel}'.format(channel=channel_2))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=dropout2d_p))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=dropout1d_p))
    print('========================================================================')
    index = (lr, dropout2d_p, dropout1d_p)
    models_conv2_linear2[index] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 128),
      nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(128, 10),
      nn.Dropout1d(dropout1d_p)
    )
    optimizer = optim.Adam(models_conv2_linear2[index].parameters(), lr=lr)

    train_part34(models_conv2_linear2[index], optimizer, epochs=2)

COMBINATION 0
lr is 0.001086694041464698
dropout2d_p is 0.09604557603801958
dropout1d_p is 0.09516395405276093
Epoch 0.00
Iteration 0, loss = 2.3245
Checking accuracy on validation set
Got 120 / 1000 correct (12.00)

Iteration 240, loss = 1.4394
Checking accuracy on validation set
Got 474 / 1000 correct (47.40)

Iteration 480, loss = 1.4841
Checking accuracy on validation set
Got 551 / 1000 correct (55.10)

Iteration 720, loss = 1.4912
Checking accuracy on validation set
Got 545 / 1000 correct (54.50)

Epoch 1.00
Iteration 0, loss = 1.5772
Checking accuracy on validation set
Got 563 / 1000 correct (56.30)

Iteration 240, loss = 1.6209
Checking accuracy on validation set
Got 611 / 1000 correct (61.10)

Iteration 480, loss = 1.1792
Checking accuracy on validation set
Got 621 / 1000 correct (62.10)

Iteration 720, loss = 1.3247
Checking accuracy on validation set
Got 658 / 1000 correct (65.80)

COMBINATION 1
lr is 0.0040309091614146725
dropout2d_p is 0.09609588329564825
dropout1d_p is 0.0

In [ ]:
for key, model in models_conv2_linear2.items():
    print('========================================================================')
    check_accuracy_part34(loader_val, model)
    # if check_accuracy_part34(loader_val, model)>=0.7:
    #   best_model_1 = model
    print('lr is {lr}'.format(lr=key[0]))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=key[1]))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=key[2]))
    print('========================================================================')

Checking accuracy on validation set
Got 659 / 1000 correct (65.90)
lr is 0.001086694041464698
dropout2d_p is 0.09604557603801958
dropout1d_p is 0.09516395405276093
Checking accuracy on validation set
Got 509 / 1000 correct (50.90)
lr is 0.0040309091614146725
dropout2d_p is 0.09609588329564825
dropout1d_p is 0.017760053523384556
Checking accuracy on validation set
Got 659 / 1000 correct (65.90)
lr is 0.0016992454183443148
dropout2d_p is 0.06594432478037378
dropout1d_p is 0.057515391258561366
Checking accuracy on validation set
Got 473 / 1000 correct (47.30)
lr is 0.003325568941192587
dropout2d_p is 0.05436966447320928
dropout1d_p is 0.038398041752476124
Checking accuracy on validation set
Got 641 / 1000 correct (64.10)
lr is 0.0013606946688283852
dropout2d_p is 0.020330383077797255
dropout1d_p is 0.09901255123989239
Checking accuracy on validation set
Got 461 / 1000 correct (46.10)
lr is 0.003988030189812957
dropout2d_p is 0.05450670537326966
dropout1d_p is 0.020605517708096068
Checking

In [ ]:
import random
print_every = 240

filter_1 = (3,3)
filter_2 = (3,3)
lr_bounds = [1e-4, 1e-3]
# channel_1_bounds = [30, 35]
# channel_2_bounds = [44, 50]
dropout2d_bounds = [0,0.1]
dropout1d_bounds = [0,0.1]
num_linear_features_opt = [64, 128, 256]
# lr = 1e-3
channel_1 = 36
channel_2 = 48
# dropout2d_bounds = 0.08
# dropout1d_bounds = 0.1
models_conv2_linear2 = dict()

for i in range(10):
    lr = np.random.uniform(lr_bounds[0], lr_bounds[1])
    # channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    # channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    dropout2d_p = np.random.uniform(dropout2d_bounds[0], dropout2d_bounds[1])
    dropout1d_p = np.random.uniform(dropout1d_bounds[0], dropout2d_bounds[1])
    num_linear_features = random.choice(num_linear_features_opt)
    print('========================================================================')
    print('COMBINATION {i}'.format(i=i))
    print('lr is {lr}'.format(lr=lr))
    print('num_linear_features is {num_linear_features}'.format(num_linear_features=num_linear_features))
    # print('channel_1 is {channel}'.format(channel=channel_1))
    # print('channel_2 is {channel}'.format(channel=channel_2))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=dropout2d_p))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=dropout1d_p))
    print('========================================================================')
    index = (lr, num_linear_features, dropout2d_p, dropout1d_p)
    models_conv2_linear2[index] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, num_linear_features),
      nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(num_linear_features, 10),
      nn.Dropout1d(dropout1d_p)
    )
    optimizer = optim.Adam(models_conv2_linear2[index].parameters(), lr=lr)

    train_part34(models_conv2_linear2[index], optimizer, epochs=4)

COMBINATION 0
lr is 0.00014227685289518657
num_linear_features is 256
dropout2d_p is 0.02534142843544589
dropout1d_p is 0.03779212145016672
Epoch 0.00
Iteration 0, loss = 2.3182
Checking accuracy on validation set
Got 96 / 1000 correct (9.60)

Iteration 240, loss = 1.4316
Checking accuracy on validation set
Got 535 / 1000 correct (53.50)

Iteration 480, loss = 1.3529
Checking accuracy on validation set
Got 589 / 1000 correct (58.90)

Iteration 720, loss = 1.1999
Checking accuracy on validation set
Got 618 / 1000 correct (61.80)

Epoch 1.00
Iteration 0, loss = 1.2312
Checking accuracy on validation set
Got 606 / 1000 correct (60.60)

Iteration 240, loss = 1.2029
Checking accuracy on validation set
Got 621 / 1000 correct (62.10)

Iteration 480, loss = 1.2782
Checking accuracy on validation set
Got 651 / 1000 correct (65.10)

Iteration 720, loss = 0.9586
Checking accuracy on validation set
Got 639 / 1000 correct (63.90)

Epoch 2.00
Iteration 0, loss = 0.9394
Checking accuracy on validatio

In [ ]:
for key, model in models_conv2_linear2.items():
    print('========================================================================')
    check_accuracy_part34(loader_val, model)
    if check_accuracy_part34(loader_val, model)>=0.7:
      best_model_1 = model
    print('lr is {lr}'.format(lr=key[0]))
    print('num_linear_features is {num_linear_features}'.format(num_linear_features=key[1]))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=key[2]))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=key[3]))
    print('========================================================================')

Checking accuracy on validation set
Got 689 / 1000 correct (68.90)
Checking accuracy on validation set
Got 689 / 1000 correct (68.90)
lr is 0.00014227685289518657
num_linear_features is 256
dropout2d_p is 0.02534142843544589
dropout1d_p is 0.03779212145016672
Checking accuracy on validation set
Got 666 / 1000 correct (66.60)
Checking accuracy on validation set
Got 666 / 1000 correct (66.60)
lr is 0.0002785002170731593
num_linear_features is 128
dropout2d_p is 0.051369049007013584
dropout1d_p is 0.05842826261403135
Checking accuracy on validation set
Got 686 / 1000 correct (68.60)
Checking accuracy on validation set
Got 686 / 1000 correct (68.60)
lr is 0.00032620168452287835
num_linear_features is 64
dropout2d_p is 0.0024206330270504454
dropout1d_p is 0.051443877239961425
Checking accuracy on validation set
Got 668 / 1000 correct (66.80)
Checking accuracy on validation set
Got 668 / 1000 correct (66.80)
lr is 0.0005286663218171006
num_linear_features is 64
dropout2d_p is 0.0971546024795

In [ ]:
train_part34(best_model_1, optimizer, epochs=6)

Epoch 0.00
Iteration 0, loss = 0.4998
Checking accuracy on validation set
Got 708 / 1000 correct (70.80)

Iteration 240, loss = 0.6407
Checking accuracy on validation set
Got 711 / 1000 correct (71.10)

Iteration 480, loss = 0.5888
Checking accuracy on validation set
Got 713 / 1000 correct (71.30)

Iteration 720, loss = 0.4542
Checking accuracy on validation set
Got 715 / 1000 correct (71.50)

Epoch 1.00
Iteration 0, loss = 0.4723
Checking accuracy on validation set
Got 705 / 1000 correct (70.50)

Iteration 240, loss = 0.6474
Checking accuracy on validation set
Got 711 / 1000 correct (71.10)

Iteration 480, loss = 0.5972
Checking accuracy on validation set
Got 712 / 1000 correct (71.20)

Iteration 720, loss = 0.5150
Checking accuracy on validation set
Got 714 / 1000 correct (71.40)

Epoch 2.00
Iteration 0, loss = 0.5782
Checking accuracy on validation set
Got 714 / 1000 correct (71.40)

Iteration 240, loss = 0.4989
Checking accuracy on validation set
Got 711 / 1000 correct (71.10)

Ite

In [ ]:
check_accuracy_part34(loader_val, best_model_1)

Checking accuracy on validation set
Got 715 / 1000 correct (71.50)


0.715

In [ ]:
PATH = '/content/drive/My Drive/{}models/best_model_conv2_linear2'.format(FOLDERNAME)
torch.save(best_model_1, PATH)

In [ ]:
print_every = 240
filter_1 = (3,3)
filter_2 = (3,3)
channel_1 = 36
channel_2 = 48
num_linear_features = 256
lr=3.8e-4

model_dropout0 = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      # nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      # nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, num_linear_features),
      # nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(num_linear_features, 10),
      # nn.Dropout1d(dropout1d_p)
    )
optimizer = optim.Adam(model_dropout0.parameters(), lr=lr)

train_part34(model_dropout0, optimizer, epochs=4)

Epoch 0.00
Iteration 0, loss = 2.3392
Checking accuracy on validation set
Got 191 / 1000 correct (19.10)

Iteration 240, loss = 1.3852
Checking accuracy on validation set
Got 549 / 1000 correct (54.90)

Iteration 480, loss = 1.2015
Checking accuracy on validation set
Got 598 / 1000 correct (59.80)

Iteration 720, loss = 0.9953
Checking accuracy on validation set
Got 624 / 1000 correct (62.40)

Epoch 1.00
Iteration 0, loss = 0.8496
Checking accuracy on validation set
Got 618 / 1000 correct (61.80)

Iteration 240, loss = 0.9700
Checking accuracy on validation set
Got 642 / 1000 correct (64.20)

Iteration 480, loss = 0.8409
Checking accuracy on validation set
Got 654 / 1000 correct (65.40)

Iteration 720, loss = 1.0144
Checking accuracy on validation set
Got 675 / 1000 correct (67.50)

Epoch 2.00
Iteration 0, loss = 0.6571
Checking accuracy on validation set
Got 680 / 1000 correct (68.00)

Iteration 240, loss = 1.0724
Checking accuracy on validation set
Got 687 / 1000 correct (68.70)

Ite

In [ ]:
check_accuracy_part34(loader_val, model_dropout0)

Checking accuracy on validation set
Got 680 / 1000 correct (68.00)


0.68

In [ ]:
train_part34(model_dropout0, optimizer, epochs=6)

Epoch 0.00
Iteration 0, loss = 0.5620
Checking accuracy on validation set
Got 688 / 1000 correct (68.80)

Iteration 240, loss = 0.8923
Checking accuracy on validation set
Got 682 / 1000 correct (68.20)

Iteration 480, loss = 0.5978
Checking accuracy on validation set
Got 688 / 1000 correct (68.80)

Iteration 720, loss = 0.3167
Checking accuracy on validation set
Got 705 / 1000 correct (70.50)

Epoch 1.00
Iteration 0, loss = 0.4338
Checking accuracy on validation set
Got 710 / 1000 correct (71.00)

Iteration 240, loss = 0.5431
Checking accuracy on validation set
Got 696 / 1000 correct (69.60)

Iteration 480, loss = 0.4262
Checking accuracy on validation set
Got 727 / 1000 correct (72.70)

Iteration 720, loss = 0.4433
Checking accuracy on validation set
Got 712 / 1000 correct (71.20)

Epoch 2.00
Iteration 0, loss = 0.2263
Checking accuracy on validation set
Got 706 / 1000 correct (70.60)

Iteration 240, loss = 0.3905
Checking accuracy on validation set
Got 711 / 1000 correct (71.10)

Ite

In [ ]:
check_accuracy_part34(loader_val, model_dropout0)

Checking accuracy on validation set
Got 722 / 1000 correct (72.20)


0.722

In [ ]:
import random
print_every = 240

filter_1 = (3,3)
filter_2 = (3,3)
lr_bounds = [1e-4, 1e-3]
# channel_1_bounds = [30, 35]
# channel_2_bounds = [44, 50]
dropout2d_bounds = [0,0.1]
dropout1d_bounds = [0,0.1]
num_linear_features_opt = [64, 128, 256]
# lr = 1e-3
channel_1 = 36
channel_2 = 48
# dropout2d_bounds = 0.08
# dropout1d_bounds = 0.1
models_conv2_2_linear2 = dict()

for i in range(10):
    lr = np.random.uniform(lr_bounds[0], lr_bounds[1])
    # channel_1 = int(np.random.uniform(channel_1_bounds[0], channel_1_bounds[1]))
    # channel_2 = int(np.random.uniform(channel_2_bounds[0], channel_2_bounds[1]))
    dropout2d_p = np.random.uniform(dropout2d_bounds[0], dropout2d_bounds[1])
    dropout1d_p = np.random.uniform(dropout1d_bounds[0], dropout2d_bounds[1])
    num_linear_features = random.choice(num_linear_features_opt)
    print('========================================================================')
    print('COMBINATION {i}'.format(i=i))
    print('lr is {lr}'.format(lr=lr))
    print('num_linear_features is {num_linear_features}'.format(num_linear_features=num_linear_features))
    # print('channel_1 is {channel}'.format(channel=channel_1))
    # print('channel_2 is {channel}'.format(channel=channel_2))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=dropout2d_p))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=dropout1d_p))
    print('========================================================================')
    index = (lr, num_linear_features, dropout2d_p, dropout1d_p)
    models_conv2_2_linear2[index] = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      # nn.MaxPool2d((2,2)),
      nn.Conv2d(channel_2, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, num_linear_features),
      nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(num_linear_features, 10),
      nn.Dropout1d(dropout1d_p)
    )
    optimizer = optim.Adam(models_conv2_2_linear2[index].parameters(), lr=lr)

    train_part34(models_conv2_2_linear2[index], optimizer, epochs=3)

COMBINATION 0
lr is 0.0007966031821950398
num_linear_features is 128
dropout2d_p is 0.0031012289010419304
dropout1d_p is 0.03459700049300918
Epoch 0.00
Iteration 0, loss = 2.3242
Checking accuracy on validation set
Got 105 / 1000 correct (10.50)

Iteration 240, loss = 1.5176
Checking accuracy on validation set
Got 502 / 1000 correct (50.20)

Iteration 480, loss = 1.2586
Checking accuracy on validation set
Got 560 / 1000 correct (56.00)

Iteration 720, loss = 1.2429
Checking accuracy on validation set
Got 630 / 1000 correct (63.00)

Epoch 1.00
Iteration 0, loss = 1.0590
Checking accuracy on validation set
Got 635 / 1000 correct (63.50)

Iteration 240, loss = 0.8531
Checking accuracy on validation set
Got 648 / 1000 correct (64.80)

Iteration 480, loss = 1.1001
Checking accuracy on validation set
Got 682 / 1000 correct (68.20)

Iteration 720, loss = 0.8351
Checking accuracy on validation set
Got 664 / 1000 correct (66.40)

Epoch 2.00
Iteration 0, loss = 0.9764
Checking accuracy on valida

In [ ]:
filter_1 = (3,3)
filter_2 = (3,3)
lr = 0.000832
dropout2d_p = 0.0437
dropout1d_p = 0.0413
num_linear_features = 128
channel_1 = 36
channel_2 = 48

models_conv2_2_linear2_best = nn.Sequential(
      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      # nn.MaxPool2d((2,2)),
      nn.Conv2d(channel_2, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, num_linear_features),
      nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(num_linear_features, 10),
      nn.Dropout1d(dropout1d_p)
    )
optimizer = optim.Adam(models_conv2_2_linear2_best.parameters(), lr=lr)
train_part34(models_conv2_2_linear2_best, optimizer, epochs=3)

Epoch 0.00
Iteration 0, loss = 2.2868
Checking accuracy on validation set
Got 112 / 1000 correct (11.20)

Iteration 100, loss = 1.9904
Checking accuracy on validation set
Got 364 / 1000 correct (36.40)

Iteration 200, loss = 1.6489
Checking accuracy on validation set
Got 439 / 1000 correct (43.90)

Iteration 300, loss = 1.7416
Checking accuracy on validation set
Got 485 / 1000 correct (48.50)

Iteration 400, loss = 1.7518
Checking accuracy on validation set
Got 510 / 1000 correct (51.00)

Iteration 500, loss = 1.1662
Checking accuracy on validation set
Got 530 / 1000 correct (53.00)

Iteration 600, loss = 1.0959
Checking accuracy on validation set
Got 581 / 1000 correct (58.10)

Iteration 700, loss = 1.3975
Checking accuracy on validation set
Got 597 / 1000 correct (59.70)

Epoch 1.00
Iteration 0, loss = 1.1312
Checking accuracy on validation set
Got 579 / 1000 correct (57.90)

Iteration 100, loss = 1.1747
Checking accuracy on validation set
Got 605 / 1000 correct (60.50)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, models_conv2_2_linear2_best)

Checking accuracy on validation set
Got 690 / 1000 correct (69.00)


0.69

In [ ]:
optimizer = optim.Adam(models_conv2_2_linear2_best.parameters(), lr=0.0000832)
train_part34(models_conv2_2_linear2_best, optimizer, epochs=7)

Epoch 0.00
Iteration 0, loss = 0.7826
Checking accuracy on validation set
Got 689 / 1000 correct (68.90)

Iteration 100, loss = 0.7302
Checking accuracy on validation set
Got 709 / 1000 correct (70.90)

Iteration 200, loss = 0.8021
Checking accuracy on validation set
Got 703 / 1000 correct (70.30)

Iteration 300, loss = 0.7549
Checking accuracy on validation set
Got 708 / 1000 correct (70.80)

Iteration 400, loss = 0.9333
Checking accuracy on validation set
Got 719 / 1000 correct (71.90)

Iteration 500, loss = 0.7799
Checking accuracy on validation set
Got 720 / 1000 correct (72.00)

Iteration 600, loss = 0.6790
Checking accuracy on validation set
Got 708 / 1000 correct (70.80)

Iteration 700, loss = 0.6594
Checking accuracy on validation set
Got 708 / 1000 correct (70.80)

Epoch 1.00
Iteration 0, loss = 0.6499
Checking accuracy on validation set
Got 722 / 1000 correct (72.20)

Iteration 100, loss = 0.8252
Checking accuracy on validation set
Got 725 / 1000 correct (72.50)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, models_conv2_2_linear2_best)

Checking accuracy on validation set
Got 743 / 1000 correct (74.30)


0.743

In [ ]:
PATH = '/content/drive/My Drive/{}models/best_model_conv2_2_linear2'.format(FOLDERNAME)
torch.save(models_conv2_2_linear2_best, PATH)

In [ ]:
optimizer = optim.Adam(models_conv2_2_linear2_best.parameters(), lr=0.000000832)
train_part34(models_conv2_2_linear2_best, optimizer, epochs=7)

Epoch 0.00
Iteration 0, loss = 0.7230
Checking accuracy on validation set
Got 739 / 1000 correct (73.90)

Iteration 100, loss = 0.8907
Checking accuracy on validation set
Got 740 / 1000 correct (74.00)

Iteration 200, loss = 0.7264
Checking accuracy on validation set
Got 741 / 1000 correct (74.10)

Iteration 300, loss = 0.7486
Checking accuracy on validation set
Got 742 / 1000 correct (74.20)

Iteration 400, loss = 0.6427
Checking accuracy on validation set
Got 740 / 1000 correct (74.00)

Iteration 500, loss = 0.8610
Checking accuracy on validation set
Got 743 / 1000 correct (74.30)

Iteration 600, loss = 0.6828
Checking accuracy on validation set
Got 740 / 1000 correct (74.00)

Iteration 700, loss = 0.8704
Checking accuracy on validation set
Got 741 / 1000 correct (74.10)

Epoch 1.00
Iteration 0, loss = 0.8621
Checking accuracy on validation set
Got 742 / 1000 correct (74.20)

Iteration 100, loss = 0.7722
Checking accuracy on validation set
Got 741 / 1000 correct (74.10)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, models_conv2_2_linear2_best)

Checking accuracy on validation set
Got 741 / 1000 correct (74.10)


0.741

In [ ]:
best_acc=0
for key, model in models_conv2_2_linear2.items():
    print('========================================================================')
    acc = check_accuracy_part34(loader_val, model)
    if acc >= best_acc:
      best_model_1 = model
      best_acc = acc
    print('lr is {lr}'.format(lr=key[0]))
    print('num_linear_features is {num_linear_features}'.format(num_linear_features=key[1]))
    print('dropout2d_p is {dropout2d_p}'.format(dropout2d_p=key[2]))
    print('dropout1d_p is {dropout1d_p}'.format(dropout1d_p=key[3]))
    print('========================================================================')

Checking accuracy on validation set
Got 688 / 1000 correct (68.80)
lr is 0.0007966031821950398
num_linear_features is 128
dropout2d_p is 0.0031012289010419304
dropout1d_p is 0.03459700049300918
Checking accuracy on validation set
Got 670 / 1000 correct (67.00)
lr is 0.0008014621183981727
num_linear_features is 64
dropout2d_p is 0.07915928196187466
dropout1d_p is 0.09504516664099848
Checking accuracy on validation set
Got 690 / 1000 correct (69.00)
lr is 0.0005697367622541192
num_linear_features is 256
dropout2d_p is 0.053150731692466924
dropout1d_p is 0.02182820100840062
Checking accuracy on validation set
Got 701 / 1000 correct (70.10)
lr is 0.00046875489014882915
num_linear_features is 128
dropout2d_p is 0.02811146991603878
dropout1d_p is 0.04983870302531396
Checking accuracy on validation set
Got 638 / 1000 correct (63.80)
lr is 0.0005490247531923978
num_linear_features is 64
dropout2d_p is 0.09232928582517035
dropout1d_p is 0.002648294615624869
Checking accuracy on validation set
G

In [ ]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0008325248478276393
    maximize: False
    weight_decay: 0
)

In [ ]:
optimizer = optim.Adam(best_model_1.parameters(), lr=0.0000832)
train_part34(best_model_1, optimizer, epochs=7)

Epoch 0.00
Iteration 0, loss = 0.8326
Checking accuracy on validation set
Got 698 / 1000 correct (69.80)

Iteration 240, loss = 0.9539
Checking accuracy on validation set
Got 733 / 1000 correct (73.30)

Iteration 480, loss = 0.9861
Checking accuracy on validation set
Got 735 / 1000 correct (73.50)

Iteration 720, loss = 0.8226
Checking accuracy on validation set
Got 728 / 1000 correct (72.80)

Epoch 1.00
Iteration 0, loss = 0.7571
Checking accuracy on validation set
Got 734 / 1000 correct (73.40)

Iteration 240, loss = 0.8230
Checking accuracy on validation set
Got 730 / 1000 correct (73.00)

Iteration 480, loss = 0.5983
Checking accuracy on validation set
Got 741 / 1000 correct (74.10)

Iteration 720, loss = 1.0834
Checking accuracy on validation set
Got 740 / 1000 correct (74.00)

Epoch 2.00
Iteration 0, loss = 0.9596
Checking accuracy on validation set
Got 734 / 1000 correct (73.40)

Iteration 240, loss = 0.7312
Checking accuracy on validation set
Got 727 / 1000 correct (72.70)

Ite

In [ ]:
optimizer = optim.Adam(best_model_1.parameters(), lr=0.000000832)
train_part34(best_model_1, optimizer, epochs=5)

Epoch 0.00
Iteration 0, loss = 0.6369
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 240, loss = 0.7304
Checking accuracy on validation set
Got 748 / 1000 correct (74.80)

Iteration 480, loss = 0.6974
Checking accuracy on validation set
Got 743 / 1000 correct (74.30)

Iteration 720, loss = 0.6748
Checking accuracy on validation set
Got 745 / 1000 correct (74.50)

Epoch 1.00
Iteration 0, loss = 0.7197
Checking accuracy on validation set
Got 748 / 1000 correct (74.80)

Iteration 240, loss = 0.5408
Checking accuracy on validation set
Got 746 / 1000 correct (74.60)

Iteration 480, loss = 0.5445
Checking accuracy on validation set
Got 746 / 1000 correct (74.60)

Iteration 720, loss = 0.6450
Checking accuracy on validation set
Got 746 / 1000 correct (74.60)

Epoch 2.00
Iteration 0, loss = 0.5877
Checking accuracy on validation set
Got 746 / 1000 correct (74.60)

Iteration 240, loss = 0.7114
Checking accuracy on validation set
Got 746 / 1000 correct (74.60)

Ite

In [ ]:
check_accuracy_part34(loader_val, best_model_1)

In [ ]:
# prompt: 3 convolutional layers with dropout + 2 fully connected layers in PyTorch

# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader
# from torchvision import datasets
# from torchvision import transforms
model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Dropout2d(0.25),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Dropout2d(0.25),
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Dropout2d(0.25),
    nn.MaxPool2d(2, 2),
    Flatten(),
    nn.Linear(128 * 16 * 16, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10),
)

optimizer = optim.Adam(model.parameters(), lr=5e-4)
train_part34(model, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 2.3153
Checking accuracy on validation set
Got 137 / 1000 correct (13.70)

Iteration 100, loss = 1.9471
Checking accuracy on validation set
Got 401 / 1000 correct (40.10)

Iteration 200, loss = 1.7533
Checking accuracy on validation set
Got 483 / 1000 correct (48.30)

Iteration 300, loss = 1.8676
Checking accuracy on validation set
Got 500 / 1000 correct (50.00)

Iteration 400, loss = 1.6468
Checking accuracy on validation set
Got 537 / 1000 correct (53.70)

Iteration 500, loss = 1.3264
Checking accuracy on validation set
Got 568 / 1000 correct (56.80)

Iteration 600, loss = 1.5705
Checking accuracy on validation set
Got 574 / 1000 correct (57.40)

Iteration 700, loss = 1.3085
Checking accuracy on validation set
Got 604 / 1000 correct (60.40)

Epoch 1.00
Iteration 0, loss = 1.2065
Checking accuracy on validation set
Got 614 / 1000 correct (61.40)

Iteration 100, loss = 0.9991
Checking accuracy on validation set
Got 613 / 1000 correct (61.30)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model)

Checking accuracy on validation set
Got 730 / 1000 correct (73.00)


0.73

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=5e-5)
train_part34(model, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 0.4576
Checking accuracy on validation set
Got 732 / 1000 correct (73.20)

Iteration 100, loss = 0.2171
Checking accuracy on validation set
Got 748 / 1000 correct (74.80)

Iteration 200, loss = 0.2834
Checking accuracy on validation set
Got 745 / 1000 correct (74.50)

Iteration 300, loss = 0.2346
Checking accuracy on validation set
Got 744 / 1000 correct (74.40)

Iteration 400, loss = 0.2975
Checking accuracy on validation set
Got 747 / 1000 correct (74.70)

Iteration 500, loss = 0.2979
Checking accuracy on validation set
Got 744 / 1000 correct (74.40)

Iteration 600, loss = 0.3729
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 700, loss = 0.2723
Checking accuracy on validation set
Got 753 / 1000 correct (75.30)

Epoch 1.00
Iteration 0, loss = 0.1574
Checking accuracy on validation set
Got 751 / 1000 correct (75.10)

Iteration 100, loss = 0.3293
Checking accuracy on validation set
Got 751 / 1000 correct (75.10)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model)

Checking accuracy on validation set
Got 751 / 1000 correct (75.10)


0.751

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=5e-6)
train_part34(model, optimizer, epochs=10)

Epoch 0.00
Iteration 0, loss = 0.1066
Checking accuracy on validation set
Got 751 / 1000 correct (75.10)

Iteration 100, loss = 0.1122
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 200, loss = 0.1192
Checking accuracy on validation set
Got 751 / 1000 correct (75.10)

Iteration 300, loss = 0.1417
Checking accuracy on validation set
Got 751 / 1000 correct (75.10)

Iteration 400, loss = 0.2759
Checking accuracy on validation set
Got 749 / 1000 correct (74.90)

Iteration 500, loss = 0.1229
Checking accuracy on validation set
Got 748 / 1000 correct (74.80)

Iteration 600, loss = 0.1879
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 700, loss = 0.3827
Checking accuracy on validation set
Got 749 / 1000 correct (74.90)

Epoch 1.00
Iteration 0, loss = 0.1620
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 100, loss = 0.1365
Checking accuracy on validation set
Got 750 / 1000 correct (75.00)

Iteration 20

In [ ]:
check_accuracy_part34(loader_val, model)

Checking accuracy on validation set
Got 752 / 1000 correct (75.20)


0.752

## Describe what you did

In the cell below you should write an explanation of what you did, any additional features that you implemented, and/or any graphs that you made in the process of training and evaluating your network.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Answer:**
1. As I am starting my training I will try the following architecture which was proposed in the last exercise:
[conv-relu-conv-relu-pool] -> [affine] -> [softmax]
parameters:
LR = 1e-2

with arbitrary modifications:
optimizer -> Adam
1st filter size - (3,3)

LR 1e-2 did not show any improvement. Probably the step is too big.

Checking 1e-3. Accuracy quickly reached 65% and then oscillated around this value while the loss was decreasing. This is a sign of overfitting. I will add regularization and search for the best learning rate.

 However the first thing to check will be a number of filters on each layer. Let's try random number of filters in 1st and 2nd layers given the bound for the 1st layer - [16, 128], 2nd layer [8, 64]. Let's have 8 random combinations of those numbers.

 After 2 epochs the best combinations of number of filters on the layers are (33, 54) and (35, 51) with accuracies of 68.3% and 68.4% respectively. Let's take a closer look and make finer search near this values. Let's try random number of filters in 1st and 2nd layers given the bound for the 1st layer - [28, 40], 2nd layer [48, 60]

 After 2 epochs the best combination of number of filters on the layers is (28, 56) with validation accuracy 69.1%. Let's stick to that architecture and try to add batchnorm to convolutional and linear layers
 With Batch normalization the accuracy of 67.6 after 10 epochs has been achieved.

 Let's try to tune learning rate and weight decay (L2 regularization). the bounds for the exponential value are [-3,-7] and [-2,-8] for learning rate and weight decay respectively.

 The model performed the best on lr, reg = (8.5e-4, 4e-4) and (9.3e-4, 6.1e-8) reaching validation accuracy of 70% and 69.7% respectively after 5 epochs.

 Let's do a narrower search near this values of reg and lr and train a model with 10 epochs.


it is clear the models overfit as after 10 epochs they show even less accuracy ~66%. Adding Dropout (p=0.5) to each layer of the model with (8.5e-4, 4e-4) lr and reg.
The accuracy of the model showed steady progress with 5 epochs and resulted in 61.4%. I will try to run the same with 10 epochs
The result is somewhat similar. Probably the regularization is too big. Let's drop weight decay.

It is obvious the model is underfitting. The loss does not decrease. Let's decrease dropout rate to 0.3. The model performed better but yet was underfitting. The loss oscillated around 1.2.

Let's decrease the dropout rate even lower. Dropout of 2d features = 0.1, dropout after linear layer = 0.3. The resulting accuracy improved to 69%.

Let's try to search for more hyperparameters: number of filters, learning rate, dropout rates and train for 4 epochs.
number of filters 1st layer: [24, 96]
number of filters 2nd layer: [12, 64]
learning rate: [1e-2, 1e-4]
dropout 2d: [0, 0.5]
dropout 1d:[0, 0.5]

the best combinations:
ACCURACY = 67.9%
=====================================
 - lr = 3.5e-3
 - channel_1 = 33
 - channel_2 = 41
 - dropout2d_p = 0.079
 - dropout1d_p = 0.13

ACCURACY = 67.0%
=====================================
 - lr = 3.2e-3
 - channel_1 = 30
 - channel_2 = 48
 - dropout2d_p = 0.056
 - dropout1d_p = 0.085

ACCURACY = 66.7%
========================================================================
 - lr = 4.1e-4
 - channel_1 = 38
 - channel_2 = 44
 - dropout2d_p = 0.146
 - dropout1d_p = 0.115


ACCURACY = 66.0%
=====================================
 - lr = 1.4e-3
 - channel_1 = 40
 - channel_2 = 32
 - dropout2d_p = 0.134
 - dropout1d_p = 0.348

ACCURACY = 65.9%
=====================================
 - lr = 1.6e-4
 - channel_1 = 62
 - channel_2 = 58
 - dropout2d_p = 0.111
 - dropout1d_p = 0.270


There is clear distinction between two sets of hyperparameters:
1. High LR~3.5e-3,  relatively low dropout rate ~0.05-0.15, num_channels_1 ~30-35, num_channels2 ~40-50
2. lower LR ~1.6e-4-1.5e-3, higher dropout rate 2d: 0.13-0.16, 1d: ~0.27-0.35
num_channels_1 ~40-60, num_channels2 ~32-60

Let's try finer search wih 10 epochs for the 1st set of ranges
lr_bounds = [1e-3, 4e-3]

channel_1_bounds = [30, 35]

channel_2_bounds = [40, 50]

dropout2d_bounds = [0.05,0.1]

dropout1d_bounds = [0.07,0.15]

I have observed an improvement in validation accuracy with the best results as below:
ACCURACY = 69.60%
========================================================================
 - lr = 1.15e-3
 - channel_1 = 34
 - channel_2 = 45
 - dropout2d_p = 0.091
 - dropout1d_p = 0.089


ACCURACY = 69.4%
=====================================
 - lr = 1.18e-3
 - channel_1 = 31
 - channel_2 = 47
 - dropout2d_p = 0.092
 - dropout1d_p = 0.080


ACCURACY = 69.3%
=====================================
 - lr = 3.7e-3
 - channel_1 = 32
 - channel_2 = 49
 - dropout2d_p = 0.085
 - dropout1d_p = 0.077


Trying even tighter range of Hyperparameters:
lr_bounds = [1e-3, 2e-3]
channel_1_bounds = [30, 35]
channel_2_bounds = [44, 50]
dropout2d_bounds = [0.07,0.09]
dropout1d_bounds = [0.08,0.12]

Got slightly better results with one model hitting 70% accuracy on a validation set.

ACCURACY = 70.0%
=====================================
 - lr = 1.04e-3
 - channel_1 = 32
 - channel_2 = 44
 - dropout2d_p = 0.079
 - dropout1d_p = 0.082


Let's try another architecture and try to find out whether we can achieve considerably better results with other numbers of convolutional or/and linear layers.

[conv-batchnorm-dropout-ReLU]x2 -> [MaxPool] -> [Affine-Dropout-ReLU] -> [softmax]

      nn.Conv2d(3, channel_1, filter_1, padding=1),
      nn.BatchNorm2d(channel_1),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.Conv2d(channel_1, channel_2, filter_2, padding=1),
      nn.BatchNorm2d(channel_2),
      nn.Dropout2d(dropout2d_p),
      nn.ReLU(),
      nn.MaxPool2d((2,2)),
      Flatten(),
      nn.Linear(channel_2*16*16, 128),
      nn.Dropout1d(dropout1d_p),
      nn.ReLU(),
      nn.Linear(128, 10),
      nn.Dropout1d(dropout1d_p)


fixing number of channels on the 1st and 2nd layer to 36 and 48 respectively.

Searching for best learning rate and dropout rates for 2d and 1d layers in the following bounds with 2 epochs:
lr_bounds = [1e-3, 5e-3]
dropout2d_bounds = [0,0.1]
dropout1d_bounds = [0,0.1]

the best accuracy was reached when lr was close to 0.1. Let's explore lower lrs by expanding the bounds to [1e-4, 1e-3] with 4 epochs

The best model hit 70.9% accuracy on validation set. dropout was very close to 0, lr = 3.9e-4, num_linear_features = 256.

To ensure this is not an outlier caused by weak regularization I have trained the model for 6 more epochs to reach an accuracy of 71.5% on a validation set! The model is saved to best_model_conv2_linear2

Since the best result was with dropout very close to zero it has almost no impact. Let's check whether we can drop it at all. After retraining the model with the lateset set of parameters the model performed even better on the validation set - 72.2% accuracy!

Let's try new architecture with more conv layers:

[conv-batchnorm-dropout-ReLU]x4 -> [MaxPool] -> [Affine-Dropout-ReLU]x2 -> [softmax]
Setting params and HPO bounds:
 - filter_1 = (3,3)
 - filter_2 = (3,3)
 - channel_1 = 36
 - channel_2 = 48
 - lr_bounds = [1e-4, 1e-3]
 - dropout2d_bounds = [0,0.1]
 - dropout1d_bounds = [0,0.1]
 - num_linear_features_opt = [64, 128, 256]

 After 3 epochs the best hyperparameters set is
  - lr = 0.000832
  - num_linear_features = 128
  - dropout2d_p = 0.0453
  - dropout1d_p = 0.0413

  with accuracy of 69.3 % after 3 epochs

  Let's continue training the same model with 10 times decreased learning rate 0.0000832 for 7 more epochs.
  The accuracy on the validation set hit 74.3%. Further decreasing the learning rate did not improve the model accuracy



## Test set -- run this only once

Now that we've gotten a result we're happy with, we test our final model on the test set (which you should store in best_model). Think about how this compares to your validation set accuracy.

In [ ]:
best_model = model
check_accuracy_part34(loader_test, best_model)

NameError: name 'model' is not defined